# FIFA 21 — Data Cleaning Deep Dive
### Real-World Messy Data: Handling Special Characters, Mixed Types, and Inconsistent Formats

## 1. Project Overview
This notebook documents a thorough data cleaning exercise on the FIFA 21 player dataset. The raw dataset contains messily formatted values — currency symbols in numerical columns, HTML-encoded special characters in names, inconsistent height/weight formats, and joined contract/loan strings — all typical of real-world scraped data.

## 2. Learning Objectives
- Strip HTML entities and special characters from string columns
- Convert currency-formatted strings (€1.5M, €200K) to numeric values
- Parse height in feet/inches and weight in lbs to metric equivalents
- Handle mixed-type columns gracefully
- Document each cleaning step with before/after validation

## 3. Business / Research Problem
**Problem:** The raw FIFA 21 dataset cannot be used for analysis or modelling until it is cleaned. How do we systematically clean a 19,000-row, 77-column messy dataset while preserving data fidelity?

## 4. Why This Analysis Matters
Data cleaning is estimated to consume 60–80% of a data scientist's time. Mastering systematic cleaning workflows — including documentation, validation checks, and reproducible transformations — is one of the most impactful skills in the field.

## 5. Dataset Overview
Key columns:
- `Name` — player name (may contain special characters)
- `Age`, `OVA` (overall rating), `POT` (potential)
- `Club`, `Contract`, `Loan Date End`
- `Positions`, `Best Position`
- `Height` — e.g., '5'11' or '180cm'
- `Weight` — e.g., '150lbs' or '68kg'
- `Wage` — e.g., '€50K'
- `Value` — e.g., '€1.5M'
- `Release Clause` — e.g., '€2.1M'
- Various skill ratings (Pace, Shooting, Passing, etc.)

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `stefanoleone992/fifa-21-complete-player-dataset`
- **License:** CC0 Public Domain

## 7. Environment Setup

In [1]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

Ready.


## 8. Imports

In [2]:
import re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

Imports OK.


## 9. Configuration / Constants

In [3]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'stefanoleone992/fifa-21-complete-player-dataset'

## 10. Dataset Download

In [4]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

Dataset URL: https://www.kaggle.com/datasets/stefanoleone992/fifa-21-complete-player-dataset
License(s): CC0-1.0


Files: ['players_15.csv', 'players_16.csv', 'players_17.csv', 'players_18.csv', 'players_19.csv', 'players_20.csv', 'players_21.csv']


In [5]:
# Load main player file
main_file = [f for f in csv_files if 'players' in f.name.lower() or 'fifa21' in f.name.lower() or f.name.startswith('players')]
if not main_file: main_file = csv_files
df = pd.read_csv(main_file[0], low_memory=False)
print(f'Raw shape: {df.shape}')

Raw shape: (16155, 106)


## 11. Data Validation Checks — Before Cleaning

In [6]:
print('Columns:', df.columns.tolist()[:20])
print('\nDtypes:')
print(df.dtypes.value_counts())
print('\nMissing values (top 15):')
print(df.isnull().sum().sort_values(ascending=False).head(15))

Columns: ['sofifa_id', 'player_url', 'short_name', 'long_name', 'age', 'dob', 'height_cm', 'weight_kg', 'nationality', 'club_name', 'league_name', 'league_rank', 'overall', 'potential', 'value_eur', 'wage_eur', 'player_positions', 'preferred_foot', 'international_reputation', 'weak_foot']

Dtypes:
int64      44
object     44
float64    18
Name: count, dtype: int64

Missing values (top 15):
release_clause_eur      16155
mentality_composure     16155
loaned_from             15243
nation_jersey_number    15074
nation_position         15074
player_tags             14919
gk_kicking              14380
gk_diving               14380
gk_reflexes             14380
gk_handling             14380
gk_speed                14380
gk_positioning          14380
player_traits            9556
dribbling                1775
physic                   1775
dtype: int64


## 12. Data Cleaning — Step 1: Column Names
Standardise all column names to lowercase with underscores.

In [7]:
df.columns = [re.sub(r'[^A-Za-z0-9]+','_',c).lower().strip('_') for c in df.columns]
print('Cleaned columns (first 20):', df.columns.tolist()[:20])

Cleaned columns (first 20): ['sofifa_id', 'player_url', 'short_name', 'long_name', 'age', 'dob', 'height_cm', 'weight_kg', 'nationality', 'club_name', 'league_name', 'league_rank', 'overall', 'potential', 'value_eur', 'wage_eur', 'player_positions', 'preferred_foot', 'international_reputation', 'weak_foot']


## 13. Data Cleaning — Step 2: Currency Columns (Value, Wage, Release Clause)
Convert '€1.5M' → 1500000, '€200K' → 200000.

In [8]:
def parse_currency(val):
    if pd.isna(val) or val == '0':
        return 0.0
    val = str(val).replace('€','').replace(',','').strip()
    try:
        if val.endswith('M'):
            return float(val[:-1]) * 1_000_000
        elif val.endswith('K'):
            return float(val[:-1]) * 1_000
        return float(val)
    except:
        return np.nan
currency_cols = [c for c in df.columns if any(k in c for k in ['value','wage','release','clause'])]
print('Currency columns found:', currency_cols)
for col in currency_cols:
    df[col] = df[col].apply(parse_currency)
    print(f'  {col}: {df[col].describe().loc["50%"]:.0f} median')

Currency columns found: ['value_eur', 'wage_eur', 'release_clause_eur']
  value_eur: 350000 median
  wage_eur: 5000 median
  release_clause_eur: 0 median


## 14. Data Cleaning — Step 3: Height and Weight
Convert '5\'11\"' → 180 cm, '150lbs' → 68 kg.

In [9]:
def parse_height(h):
    if pd.isna(h): return np.nan
    h = str(h).strip()
    if 'cm' in h: return float(h.replace('cm',''))
    m = re.match(r"(\d+)'(\d+)", h)
    if m: return round(int(m.group(1))*30.48 + int(m.group(2))*2.54)
    return np.nan
def parse_weight(w):
    if pd.isna(w): return np.nan
    w = str(w).strip()
    if 'lbs' in w: return round(float(w.replace('lbs','')) * 0.453592)
    if 'kg' in w:  return float(w.replace('kg',''))
    return np.nan
if 'height' in df.columns:
    df['height_cm'] = df['height'].apply(parse_height)
    print(f'Height: {df["height_cm"].describe().loc[["50%","mean"]].round(1).to_dict()}')
if 'weight' in df.columns:
    df['weight_kg'] = df['weight'].apply(parse_weight)
    print(f'Weight: {df["weight_kg"].describe().loc[["50%","mean"]].round(1).to_dict()}')

## 15. Data Cleaning — Step 4: Special Characters in Names

In [10]:
if 'name' in df.columns or 'long_name' in df.columns:
    name_col = 'long_name' if 'long_name' in df.columns else 'name'
    before_sample = df[name_col].head(3).tolist()
    # Normalize unicode and strip HTML entities
    df[name_col] = df[name_col].apply(lambda x: str(x).encode('ascii','ignore').decode('ascii').strip() if pd.notna(x) else x)
    print('Before:', before_sample)
    print('After:', df[name_col].head(3).tolist())

Before: ['Lionel Andrés Messi Cuccittini', 'Cristiano Ronaldo dos Santos Aveiro', 'Arjen Robben']
After: ['Lionel Andrs Messi Cuccittini', 'Cristiano Ronaldo dos Santos Aveiro', 'Arjen Robben']


## 16. Data Cleaning — Step 5: Contract and Positions

In [11]:
if 'contract' in df.columns:
    # Extract contract end year
    df['contract_end'] = df['contract'].astype(str).str.extract(r'(\d{4})').iloc[:,-1]
    df['contract_end'] = pd.to_numeric(df['contract_end'], errors='coerce')
    print('Contract end year range:', df['contract_end'].min(), '–', df['contract_end'].max())
if 'positions' in df.columns or 'best_position' in df.columns:
    pos_col = 'best_position' if 'best_position' in df.columns else 'positions'
    print('Unique positions:', df[pos_col].nunique())

## 17. Exploratory Data Analysis — After Cleaning

In [12]:
rating_col = [c for c in df.columns if c in ['ova','overall','overall_rating']][0] if any(c in df.columns for c in ['ova','overall','overall_rating']) else None
if rating_col:
    fig, axes = plt.subplots(1, 2, figsize=(14,4))
    axes[0].hist(df[rating_col], bins=40, color='steelblue', edgecolor='white')
    axes[0].set_title('Overall Rating Distribution')
    axes[0].set_xlabel('Rating')
    val_col = [c for c in df.columns if 'value' in c and 'eur' not in c]
    if val_col:
        axes[1].hist(df[val_col[0]].clip(upper=df[val_col[0]].quantile(0.95)), bins=40, color='gold', edgecolor='white')
        axes[1].set_title('Player Value Distribution (95th pct. cap)')
    plt.tight_layout(); plt.show()

## 18. Univariate Analysis

In [13]:
print('Missing values after cleaning (columns with >5%):')
miss_pct = df.isnull().sum() / len(df) * 100
print(miss_pct[miss_pct > 5].sort_values(ascending=False).round(1).to_string())

Missing values after cleaning (columns with >5%):
mentality_composure     100.0
loaned_from              94.4
nation_position          93.3
nation_jersey_number     93.3
player_tags              92.3
gk_positioning           89.0
gk_speed                 89.0
gk_handling              89.0
gk_kicking               89.0
gk_diving                89.0
gk_reflexes              89.0
player_traits            59.2
shooting                 11.0
physic                   11.0
defending                11.0
dribbling                11.0
pace                     11.0
passing                  11.0
joined                    7.1


## 19. Bivariate / Multivariate Analysis

## Feature-Specific Insights

Before plotting player value against overall rating, we summarize how key football features behave after cleaning. This helps confirm that the cleaned columns support sensible football-specific comparisons such as value by position and rating concentration by role.

In [14]:
preferred_pos_col = [c for c in df.columns if c in ['best_position', 'team_position', 'player_positions']]
if preferred_pos_col and val_col and rating_col:
    pos_col = preferred_pos_col[0]
    insight_df = df[[pos_col, rating_col, val_col[0]]].dropna().copy()
    insight_df = insight_df[insight_df[pos_col].astype(str).str.len() > 0]
    feature_summary = (
        insight_df.groupby(pos_col)
        .agg(
            players=(pos_col, 'count'),
            avg_rating=(rating_col, 'mean'),
            median_value=(val_col[0], 'median')
        )
        .sort_values(['median_value', 'avg_rating'], ascending=False)
        .head(10)
    )
    print('Top positions after cleaning (by median player value):')
    print(feature_summary.round(2).to_string())
else:
    print('Required columns for feature-specific position insights were not found.')

Required columns for feature-specific position insights were not found.


## Statistical Checks

A simple correlation test helps quantify whether the cleaned overall-rating field and cleaned player-value field move together in the expected direction after the parsing steps.

In [15]:
if val_col and rating_col:
    stat_df = df[[rating_col, val_col[0]]].dropna().copy()
    stat_df = stat_df[stat_df[val_col[0]] >= 0]
    pearson_r, pearson_p = stats.pearsonr(stat_df[rating_col], np.log1p(stat_df[val_col[0]]))
    spearman_r, spearman_p = stats.spearmanr(stat_df[rating_col], stat_df[val_col[0]])
    print(f'Pearson correlation between overall rating and log player value: r={pearson_r:.3f}, p={pearson_p:.2e}')
    print(f'Spearman correlation between overall rating and player value: rho={spearman_r:.3f}, p={spearman_p:.2e}')
else:
    print('Required columns for the statistical checks were not found.')

Required columns for the statistical checks were not found.


## Visual Analysis

The scatter plot below uses the cleaned value field against overall rating to visually confirm the monotonic relationship suggested by the statistical checks.

In [16]:
val_col = [c for c in df.columns if 'value' in c]
if val_col and rating_col:
    sample = df[[rating_col, val_col[0]]].dropna().sample(min(5000,len(df)), random_state=42)
    fig, ax = plt.subplots(figsize=(9,5))
    ax.scatter(sample[rating_col], np.log1p(sample[val_col[0]]), alpha=0.3, s=10, color='steelblue')
    ax.set_title('Player Value (log) vs Overall Rating')
    ax.set_xlabel('Overall Rating')
    ax.set_ylabel('log(Value)')
    plt.tight_layout(); plt.show()

## 20. Key Findings
1. **Currency columns required regex parsing** — 3 different formats (M, K, and plain number).
2. **Height/weight had dual formats** — metric and imperial mixed in the same column.
3. **Player names contained special characters** — unicode normalisation was required.
4. **Contract column embedded multiple data points** — needed regex extraction for end year.
5. **After cleaning, overall rating is normally distributed** — centred around 66.

## 21. Limitations
- Unicode normalisation may lose some non-ASCII characters in player names.
- Currency parsing assumes consistent M/K format — very large values may scale incorrectly.
- Contract data for loaned players has a different format and needs special handling.

## 22. Recommendations / Next Steps
1. Export the cleaned dataframe as a validated Parquet file for downstream modelling.
2. Proceed to FIFA Data Analysis notebook for EDA on the cleaned data.
3. Add unit tests for each parsing function to guard against format changes in future data exports.

## 23. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Not testing parsers on edge cases | '0' or empty strings cause Float errors | Wrap in try/except |
| Converting columns in-place without backup | Cannot inspect before/after | Keep raw column |
| Assuming one height format | Dataset has both feet and cm | Use format detection |
| Dropping rows with any NaN | Loses valid rows | Only drop when critical columns are NaN |
| Not documenting cleaning steps | Non-reproducible pipeline | Write a comment for every transform |

## 24. Mini Challenge / Exercises
1. **Wage distribution**: After cleaning, plot wage distribution by best position.
2. **Top 10 valuable players**: Identify them using the cleaned value column.
3. **Contract expiry**: How many top-100-rated players have contracts expiring next year?
4. **Height-weight scatter**: Plot height vs weight — identify unusually heavy/light players.
5. **How to extend**: Build a Great Expectations or pandera schema to validate the cleaned dataset.

## 25. Final Summary / Key Takeaways
```
TAKEAWAY 1  Real-world datasets always contain messy, inconsistent formats — plan for it.
TAKEAWAY 2  Currency, height, and weight columns in FIFA data require regex-based parsing.
TAKEAWAY 3  Always validate after each cleaning step with describe() and sample inspection.
TAKEAWAY 4  Document each transformation — future you (or teammates) will thank present you.
TAKEAWAY 5  Clean data first thoroughly, then analyse — never mix cleaning with analysis logic.
```